# Phase 9: Student-Grouped Train / Validation / Test Splitting

**Study**: *Student Performance Prediction Using Machine Learning and Explainable AI: An Experimental Study Using the Open University Learning Analytics Dataset (OULAD)*

### Methodological Rules for Phase 9:
1. **Student-Level Grouping**: Grouped strictly on `id_student` (no student appears across multiple partitions).
2. **Identical Partitions Across Scenarios**: Mapping generated once and applied identically to $S_0$, $S_1$, and $S_2$.
3. **Untouched Test Holdout (15%)**: Strictly locked and quarantined from all preprocessing, imputation, and tuning.
4. **Development Population (85%)**: 70% Train, 15% Validation, and 5-fold cross-validation fold assignments.
5. **Zero-Leakage Verification**: Programmatic check confirms `intersection(Train, Val, Test) == 0`.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
SPLITS_DIR = os.path.join(PROJECT_ROOT, "results", "splits")
TABLES_DIR = os.path.join(PROJECT_ROOT, "results", "tables")

if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from data_splitting import create_student_grouped_partitions

partition_map, summary = create_student_grouped_partitions(PROCESSED_DIR, SPLITS_DIR, TABLES_DIR)
display(pd.DataFrame(summary))

In [ ]:
# Verify zero overlap
train_ids = set(partition_map[partition_map['partition'] == 'train']['id_student'])
val_ids = set(partition_map[partition_map['partition'] == 'validation']['id_student'])
test_ids = set(partition_map[partition_map['partition'] == 'test']['id_student'])

print(f"Train Unique Students: {len(train_ids):,}")
print(f"Validation Unique Students: {len(val_ids):,}")
print(f"Test Unique Students: {len(test_ids):,}")
print(f"Overlap Train-Val: {len(train_ids.intersection(val_ids))}")
print(f"Overlap Train-Test: {len(train_ids.intersection(test_ids))}")
print(f"Overlap Val-Test: {len(val_ids.intersection(test_ids))}")

assert len(train_ids.intersection(val_ids)) == 0
assert len(train_ids.intersection(test_ids)) == 0
assert len(val_ids.intersection(test_ids)) == 0
print("\nALL GROUP LEAKAGE VERIFICATION CHECKS PASSED!")